In [1]:
print(10/0)

ZeroDivisionError: division by zero

In [3]:
try :
    result = 10/0
except ValueError :
    print("Cannot divide by zero")

ZeroDivisionError: division by zero

In [9]:
import pandas as pd
try :
    df = pd.read_csv("sales_data.csv")
except FileNotFoundError as e :
    print(f'caught : {e}')
else :
    print("file load completed",df.shape)
finally:
    print("Attempt completed")


file load completed (10, 9)
Attempt completed


In [10]:
try:
    df = pd.read_csv("fake_file.csv")
except FileNotFoundError as e:
    print(f"Error : {e}")
else:
    print(f"file loaded : {df.shape}")
finally:
    print("Attempt complete")

Error : [Errno 2] No such file or directory: 'fake_file.csv'
Attempt complete


In [20]:
def read_files(filepath,encoding = 'utf-8',nrows = None):
    try :
        df = pd.read_csv(filepath, encoding = encoding , nrows= nrows)
        print(f'loaded {filepath} - {df.shape}')
        return df
    except FileNotFoundError :
        print(f'file not found: {filepath}')
        #return None
    except UnicodeDecodeError:
        print(f'Encoding Error - try encoding = Latin 1')
        return None
    except Exception as e :
        print(f'Unexpected Error : {e}')
        return None
    print("Function executed fully")
    


sales_df = read_files("fake_file.csv")
        
    

file not found: fake_file.csv
Function executed fully


In [21]:
def read_files(filepath):
    try:
        df = pd.read_csv(filepath)
        return df
    except FileNotFoundError:
        print("File not found")
        # no return

    print("I still executed!")   # does this run?

read_files('fake_file.csv')

File not found
I still executed!


In [22]:
def read_files(filepath, encoding='utf-8', nrows=None):
    try:
        df = pd.read_csv(filepath, encoding=encoding, nrows=nrows)
        print(f"Loaded {filepath} — {df.shape}")
        return df
    except FileNotFoundError:
        print(f"File not found: {filepath}")
        return None
    except UnicodeDecodeError:
        print(f"Encoding error — try encoding='latin-1'")
        return None
    except Exception as e:
        print(f"Unexpected error: {e}")
        return None

def clean_file(df, fill_value = 0, drop_col = None):
    try:
        df = df.fillna(fill_value)
        if drop_col :
            df = df.drop(columns = drop_col)
            df =  df.reset_index(drop = True)
        return df 
    except KeyError :
        print(f'coloumn not found:{drop_col}')
        return None
    except Exception as e :
        print(f'Unknown exception occured:{e}')
        return None
            
         

# Test both
df1 = read_files('sales_data.csv')
print(type(df1))

print("---")

df2 = read_files('fake_file.csv')
print(type(df2))

Loaded sales_data.csv — (10, 9)
<class 'pandas.DataFrame'>
---
File not found: fake_file.csv
<class 'NoneType'>


In [28]:
def read_files(filepath, encoding='utf-8', nrows=None):
    try:
        df = pd.read_csv(filepath, encoding=encoding, nrows=nrows)
        print(f"Loaded {filepath} — {df.shape}")
        return df
    except FileNotFoundError:
        print(f"File not found: {filepath}")
        return None
    except UnicodeDecodeError:
        print(f"Encoding error — try encoding='latin-1'")
        return None
    except Exception as e:
        print(f"Unexpected error: {e}")
        return None

def clean_file(df, fill_value = 0, drop_col = None):
    try:
        df = df.fillna(fill_value)
        if drop_col :
            df = df.drop(columns = drop_col)
            df =  df.reset_index(drop = True)
        return df 
    except KeyError :
        print(f'coloumn not found:{drop_col}')
        return None
    except Exception as e :
        print(f'Unknown exception occured:{e}')
        return None

def transform_data(df, group_col, value_col,agg_func = "sum") :
    try :
         result = df.groupby(group_col)[value_col].agg(agg_func).reset_index()
         result = result.rename(columns = {value_col : f'{agg_func}_{value_col}'})
         return result
    except Exception as e :
        print(f'exception {e} has occured')
        return None
        
   
            


def run_pipeline(filepath, group_col, value_col,
                 agg_func='sum', fill_value=0, drop_col=None):

    # Step 1 — Read
    df = read_files(filepath)
    if df is None:
        print("Pipeline stopped — file could not be loaded")
        return None

    # Step 2 — Clean
    df = clean_file(df, fill_value=fill_value, drop_col=drop_col)
    if df is None:
        print("Pipeline stopped — cleaning failed")
        return None

    # Step 3 — Transform
    result = transform_data(df, group_col, value_col, agg_func)
    if result is None:
        print("Pipeline stopped — transform failed")
        return None

    print("Pipeline completed successfully")
    return result

# Scenario 1 — real file
output1 = run_pipeline('sales_data.csv', 'region', 'sales')
print(output1)

print('\n')
# Scenario 2 — fake file
output2 = run_pipeline('fake_file.csv', 'region', 'sales')
print(output2)
print('\n')
output3 = run_pipeline('sales_data.csv', 'region', 'sales', drop_col='nonexistent')
print(output3)

Loaded sales_data.csv — (10, 9)
Pipeline completed successfully
  region  sum_sales
0  North      56000
1  South      40000
2   West      60000


File not found: fake_file.csv
Pipeline stopped — file could not be loaded
None


Loaded sales_data.csv — (10, 9)
coloumn not found:nonexistent
Pipeline stopped — cleaning failed
None


In [1]:
import logging

# Configure logging
logging.basicConfig(
    level  = logging.INFO, #info level is 20. anything level 20 won't be appear in logfile
    format = '%(asctime)s - %(levelname)s - %(message)s',
    handlers = [
        logging.FileHandler('pipeline.log'),
        logging.StreamHandler()
    ]
)

logger = logging.getLogger(__name__)

# Test all four levels
logger.debug("This is debug")
logger.info("This is info")
logger.warning("This is warning")
logger.error("This is error")

2026-08-08 09:21:31,447 - INFO - This is info
2026-08-08 09:21:31,453 - WARNING - This is warning
2026-08-08 09:21:31,458 - ERROR - This is error


#Final complete Code with Error Handling and logging


In [10]:
import pandas as pd
import logging


"""
DEBUG    → level 10  
INFO     → level 20  
WARNING  → level 30  
ERROR    → level 40  
CRITICAL → level 50  

if you set level = INFO then all the levels below INFO that is DEBUG 
won't show up in the log file

"""
#configure the logging

logging.basicConfig(
    level = logging.INFO,
    format = '%(asctime)s - %(levelname)s - %(message)s',
    handlers = [
        logging.FileHandler('pipeline.log'),
        logging.StreamHandler()
    ]
)

logger = logging.getLogger(__name__)



def read_files(filepath, encoding ='utf-8',nrows = None) :
    try :
        df = pd.read_csv(filepath,encoding = encoding , nrows = nrows)
        logger.info(f"loaded {filepath} - {df.shape}")
        return df
    except FileNotFoundError :
        logger.error(f"file not found : {filepath}")
        return None
    except UnicodeDecodeError :
        logger.error(f"encoding error - try encoding = 'Latin-1'")
        return None
    except Exception as e:
        logger.error(f"Unexpected error : {e}")
        return None

def clean_file(df,fill_value = 0 , drop_col = None):
    try:
        df = df.fillna(fill_value)
        if drop_col :
            df = df.drop(columns = drop_col).reset_index(drop = True)
        logger.info(f"Cleaning complete - shape : {df.shape}")
        return df
    except KeyError as e :
        logger.error(f"column not found: {e}")
        return None
    except Exception as e :
        logger.error(f"cleaning failed: {e}")
        return None

def transform_data(df,group_col,value_col,agg_func = "sum"):
    try:
        result = df.groupby(group_col)[value_col].agg(agg_func).reset_index()
        result = result.rename(columns = {value_col : f"{agg_func}_{value_col}"})
        logger.info(f"transform complete - {group_col} by {agg_func}_{value_col}")
        return result
    except Exception as e:
        logger.error(f"transform failed: {e}")
        return None

def run_pipeline(filepath, group_col, value_col,
                 agg_func='sum', fill_value=0, drop_col=None):

    # Step 1 — Read
    df = read_files(filepath)
    if df is None:
        logger.error(f"there is an error while reading the file - Pipeline stopped — file could not be loaded")
        return None

    # Step 2 — Clean
    df = clean_file(df, fill_value=fill_value, drop_col=drop_col)
    if df is None:
        logger.error(f"there is an error while cleaning the data-Pipeline stopped — cleaning failed")
        return None

    # Step 3 — Transform
    result = transform_data(df, group_col, value_col, agg_func)
    if result is None:
        logger.error(f"there is an error while transforming the file-Pipeline stopped — transform failed")
        return None

    logger.info(f"pipe line completed successfully")
    return result
        
output1 = run_pipeline('sales_data.csv', 'region', 'sales')
print(output1)      




2026-08-08 18:44:35,868 - INFO - loaded sales_data.csv - (10, 9)
2026-08-08 18:44:35,873 - INFO - Cleaning complete - shape : (10, 9)
2026-08-08 18:44:35,884 - INFO - transform complete - region by sum_sales
2026-08-08 18:44:35,887 - INFO - pipe line completed successfully


  region  sum_sales
0  North      56000
1  South      40000
2   West      60000
